 # AI Ruby on Rails Code Review

 This agent helps to review pull requests in minutes, it suggest best practices, coding convention, security vectors and linters. 

# Install dependencies

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


# Imports

In [2]:
from src.github import download_pr
from src.rag import load_vector_db, retrieve
from src.reviewer import review
from src.report import markdown_report
from src.parser import parse_diff

/Users/renzodiaz/.local/share/mise/installs/python/3.14.6/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load DB

In [3]:
db = load_vector_db()

print(f"RAG database is ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20863.15it/s]


RAG database is ready.


# Download Pull Request

In [ ]:
PR_URL = "https://github.com/renzodiaz/notes-api/pull/4"

diff = download_pr(PR_URL)

print(f"Downloaded {len(diff)} characters.")

Downloaded 5625 characters.


# Parse the Pull Request

In [5]:
parsed = parse_diff(diff)

print("Ruby files:")
for filename in parsed["ruby_files"]:
    print(f"-", filename)

Ruby files:
- app/controllers/api/v1/auth_controller.rb
- app/controllers/api/v1/secure_controller.rb
- app/models/user.rb
- config/routes.rb
- db/migrate/20260808154603_create_users.rb
- db/schema.rb
- test/models/user_test.rb


# Review & Print

In [ ]:
review_result = review(diff=diff, db=db)

print(review_result)

# Overall Review

## Summary

This PR adds a basic `User` model with password support and an API login endpoint. The intent is good, but there are a few correctness and security issues in the auth flow, plus some missing database and test safeguards that should be addressed before merging.

## Issues

### [High] Login action always returns unauthorized, even on successful authentication

Category:
Rails / Maintainability

Explanation:
In `Api::V1::AuthController#login`, the success branch renders the user, but execution continues and the unauthorized response is rendered immediately afterward:

```ruby
if user && user.authenticate(params[:password])
  render json: { user: user }, status: :ok
end

render json: { error: "Invalid email or password" }, status: :unauthorized
```

This means the action will always attempt to render twice, which will raise a `DoubleRenderError` in Rails. As written, valid login requests will fail.

Recommendation:
Return early after the success render, or use